# 3d — Stacking avec apprenants à préparations mixtes (CRISP-DM Phase 4)

Le **stacking** combine plusieurs modèles de base hétérogènes : chaque base learner fait sa propre prédiction sur `X_train` (via cross-fitting interne pour éviter la fuite), puis un **méta-modèle** apprend à pondérer ces prédictions optimalement.

### Spécificité de notre stacking

Chaque base learner provient d'une **famille différente** et utilise donc son **propre préprocesseur** :

| Base learner       | Préprocesseur        | Vient de  | Pourquoi le choisir                                       |
|--------------------|----------------------|-----------|-----------------------------------------------------------|
| **Lasso**          | `preprocessor_scaled` | 3a       | Champion intra-famille des modèles linéaires régularisés  |
| **GradientBoosting** | `preprocessor_encoded` | 3b      | Meilleur arbre sklearn (boosting > bagging ici)            |
| **XGBoost**        | `preprocessor_native` | 3c       | Meilleure performance globale grâce à la gestion native    |

Le méta-modèle est **`RidgeCV`** — un linéaire pénalisé pour combiner les sorties des 3 base learners (1 prédiction par learner → 3 features pour le méta-modèle).

### Hypothèse principale du stacking
Les base learners produisent des **erreurs décorrélées** : si tous se trompent dans la même direction sur les mêmes observations, le stacking ne peut rien tirer de plus. On vérifiera la matrice de corrélation des résidus à la fin.

In [ ]:
%load_ext autoreload
%autoreload 2
%run 2_data_prep.ipynb

In [ ]:
from sklearn.ensemble import StackingRegressor, GradientBoostingRegressor
from sklearn.linear_model import LassoCV, RidgeCV
from xgboost import XGBRegressor

FAMILY = 'stacking'

## 4.1 Construction des trois sub-pipelines

`StackingRegressor` accepte une liste d'`(name, estimator)` où chaque estimator peut être un `Pipeline` à part entière (avec son propre préprocesseur). C'est ainsi qu'on combine des base learners qui n'utilisent pas le même prétraitement.

In [ ]:
# Lasso sub-pipeline (preprocessor_scaled)
lasso_sub = Pipeline([
    ('preprocessor', preprocessor_scaled),
    ('model', LassoCV(alphas=np.logspace(-4, 2, 20), cv=5, max_iter=50_000, random_state=RANDOM_STATE)),
])

# GradientBoosting sub-pipeline (preprocessor_encoded)
gb_sub = Pipeline([
    ('preprocessor', preprocessor_encoded),
    ('model', GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE)),
])

# XGBoost sub-pipeline (preprocessor_native)
xgb_sub = Pipeline([
    ('preprocessor', preprocessor_native),
    ('model', XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=4,
        enable_categorical=True, tree_method='hist',
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
    )),
])

print("Base learners construits :")
print(f"  - Lasso     (preprocessor_scaled)")
print(f"  - GBR       (preprocessor_encoded)")
print(f"  - XGBoost   (preprocessor_native)")

## 4.2 Entraînement du stacking

In [ ]:
t0 = time.time()
stack_model = StackingRegressor(
    estimators=[
        ('lasso', lasso_sub),
        ('gbr', gb_sub),
        ('xgb', xgb_sub),
    ],
    final_estimator=RidgeCV(alphas=np.logspace(-2, 2, 10)),
    cv=5,
    n_jobs=-1,
    passthrough=False,
)
stack_model.fit(X_train, y_train_log)
stack_fit_s = time.time() - t0

stack_pred = stack_model.predict(X_test)
stack_holdout = rmsle_score(y_test_log, stack_pred)

# Note: pas de CV externe ici — StackingRegressor fait déjà son propre cross-fitting
# interne (cv=5) pour générer les out-of-fold predictions du méta-modèle. Une CV externe
# ferait du nested-CV (25 fits) sans gain informatif, et déclenche des warnings sur les
# folds où certaines colonnes rares (ex. PoolQC) sont entièrement NaN en train.
stack_cv = None

print(f"Stacking — Holdout RMSLE: {stack_holdout:.4f}   fit_time: {stack_fit_s:.1f}s")
print(f"  (CV interne via StackingRegressor cv=5 — pas de CV externe ici)")
predicted_vs_actual_plot(y_test_log, stack_pred, title=f"Stacking (Lasso + GBR + XGB → RidgeCV) — RMSLE: {stack_holdout:.4f}")
plt.show()

# Inspect meta-learner weights — they tell us how much each base learner contributes
meta = stack_model.final_estimator_
weights = pd.DataFrame({'base_learner': ['Lasso', 'GBR', 'XGB'], 'meta_weight': meta.coef_})
print("\nPoids du méta-modèle (RidgeCV) :")
display(weights)

publish_result(FAMILY, 'Stacking', cv_rmsle=stack_cv, holdout_rmsle=stack_holdout, fit_time_s=stack_fit_s,
               params={'meta': 'RidgeCV', 'base': ['Lasso', 'GBR', 'XGB']},
               notes=f"Meta weights: Lasso={meta.coef_[0]:.3f}, GBR={meta.coef_[1]:.3f}, XGB={meta.coef_[2]:.3f}")

## 4.3 Diagnostic : décorrélation des base learners

Pour que le stacking apporte de la valeur, il faut que les erreurs des base learners soient **décorrélées** (sinon on combine les mêmes erreurs). On extrait les prédictions OOF (out-of-fold) que chaque base learner fournit au méta-modèle, et on regarde leur matrice de corrélation.

In [ ]:
# Re-fit each sub-pipeline standalone on X_train to obtain holdout predictions for diagnostics
lasso_sub.fit(X_train, y_train_log)
gb_sub.fit(X_train, y_train_log)
xgb_sub.fit(X_train, y_train_log)

preds_df = pd.DataFrame({
    'Lasso':   lasso_sub.predict(X_test),
    'GBR':     gb_sub.predict(X_test),
    'XGB':     xgb_sub.predict(X_test),
    'y_true':  y_test_log.values,
})

# Residuals
residuals = preds_df[['Lasso', 'GBR', 'XGB']].sub(preds_df['y_true'], axis=0)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(residuals.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Corrélation des résidus entre base learners")
plt.tight_layout()
plt.show()

print("Interprétation :")
print(" - Corrélation proche de 1 → les modèles font les mêmes erreurs, stacking peu utile.")
print(" - Corrélation faible / différentes signes → erreurs complémentaires, stacking exploite la diversité.")

## 4.4 Publication du résultat de famille

In [ ]:
family_path = RESULTS_DIR / f'family_{FAMILY}.json'
fam = pd.DataFrame(json.loads(family_path.read_text()))
display(fam[['model', 'cv_rmsle', 'holdout_rmsle', 'fit_time_s', 'notes']])